# Chapter 2 Practical 02: TF-IDF Movie Recommender

Learning objectives:
- Clean and combine text metadata.
- Build Bag-of-Words and TF-IDF item vectors.
- Compute a cosine similarity matrix.
- Explain recommendations with shared terms.

Slide connection: Bag-of-Words, TF-IDF, vector normalization, cosine similarity, ranking, and Top-N recommendation.


Load the small Chapter 2 movie dataset. It includes titles, genres, descriptions, and keywords.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


,movie_id,title,genres,director,year,duration_min,rating,family_friendly,description,keywords
0,1,Inception,Sci-Fi|Thriller|Action,Christopher Nolan,2010,148,8.8,0,A thief enters layered dreams to plant an idea...,dreams heist subconscious mind-bending
1,2,Interstellar,Sci-Fi|Adventure|Drama,Christopher Nolan,2014,169,8.7,0,Astronauts travel through a wormhole to find a...,space exploration wormhole survival family
2,3,Titanic,Romance|Drama,James Cameron,1997,195,7.9,0,A young couple from different social classes f...,romance ship tragedy historical
3,4,The Matrix,Sci-Fi|Action,The Wachowskis,1999,136,8.7,0,A hacker discovers that reality is a simulated...,simulation hacker reality action cyberpunk
4,5,Toy Story,Animation|Adventure|Comedy|Family,John Lasseter,1995,81,8.3,1,A cowboy doll feels threatened when a space ra...,toys friendship family adventure


We combine several text fields. This gives the recommender more content evidence than title or genre alone.


In [2]:
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

movies["combined_text"] = (
    movies["title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["director"] + " " +
    movies["description"] + " " +
    movies["keywords"]
).apply(clean_text)

movies[["title", "combined_text"]].head()


,title,combined_text
0,Inception,inception sci-fi thriller action christopher n...
1,Interstellar,interstellar sci-fi adventure drama christophe...
2,Titanic,titanic romance drama james cameron a young co...
3,The Matrix,the matrix sci-fi action the wachowskis a hack...
4,Toy Story,toy story animation adventure comedy family jo...


Bag-of-Words counts words. Common words can dominate because each word is weighted mostly by frequency.


In [3]:
count_vectorizer = CountVectorizer(stop_words="english")
bow_matrix = count_vectorizer.fit_transform(movies["combined_text"])

bow_preview = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=movies["title"],
)
bow_preview.iloc[:5, :12]


,action,actor,adventure,alfonso,ambition,andrew,animation,artistic,aspiring,astronaut,astronauts,barriers
title,,,,,,,,,,,,
Inception,1,0,0,0,0,0,0,0,0,0,0,0
Interstellar,0,0,1,0,0,0,0,0,0,0,1,0
Titanic,0,0,0,0,0,0,0,0,0,0,0,0
The Matrix,2,0,0,0,0,0,0,0,0,0,0,0
Toy Story,0,0,2,0,0,0,1,0,0,0,0,0


TF-IDF lowers the weight of terms that appear in many movies and raises distinctive terms.


In [4]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(movies["combined_text"])

tfidf_preview = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=movies["title"],
)
tfidf_preview.iloc[:5, :12].round(2)


,action,actor,adventure,alfonso,ambition,andrew,animation,artistic,aspiring,astronaut,astronauts,barriers
title,,,,,,,,,,,,
Inception,0.18,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0
Interstellar,0.00,0.0,0.16,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.23,0.0
Titanic,0.00,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0
The Matrix,0.34,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0
Toy Story,0.00,0.0,0.27,0.0,0.0,0.0,0.19,0.0,0.0,0.0,0.00,0.0


The similarity matrix compares every movie with every other movie.


In [5]:
bow_similarity = cosine_similarity(bow_matrix)
tfidf_similarity = cosine_similarity(tfidf_matrix)

pd.DataFrame(tfidf_similarity, index=movies["title"], columns=movies["title"]).round(2)


title,Inception,Interstellar,Titanic,The Matrix,Toy Story,Finding Nemo,The Dark Knight,The Martian,The Notebook,Paddington,Gravity,La La Land
title,,,,,,,,,,,,
Inception,1.00,0.12,0.00,0.10,0.00,0.00,0.10,0.04,0.00,0.00,0.09,0.07
Interstellar,0.12,1.00,0.02,0.04,0.17,0.08,0.09,0.13,0.04,0.15,0.20,0.02
Titanic,0.00,0.02,1.00,0.00,0.00,0.08,0.02,0.00,0.31,0.04,0.02,0.20
The Matrix,0.10,0.04,0.00,1.00,0.00,0.00,0.06,0.04,0.00,0.00,0.05,0.00
Toy Story,0.00,0.17,0.00,0.00,1.00,0.17,0.00,0.09,0.04,0.24,0.03,0.00
Finding Nemo,0.00,0.08,0.08,0.00,0.17,1.00,0.00,0.09,0.00,0.16,0.00,0.00
The Dark Knight,0.10,0.09,0.02,0.06,0.00,0.00,1.00,0.00,0.04,0.00,0.02,0.02
The Martian,0.04,0.13,0.00,0.04,0.09,0.09,0.00,1.00,0.00,0.08,0.23,0.00
The Notebook,0.00,0.04,0.31,0.00,0.04,0.00,0.04,0.00,1.00,0.00,0.04,0.23


This function returns Top-N similar movies and shows which TF-IDF terms are shared with the input movie.


In [6]:
def shared_terms(input_idx, other_idx, matrix, vectorizer, top_terms=6):
    feature_names = np.array(vectorizer.get_feature_names_out())
    input_weights = matrix[input_idx].toarray().ravel()
    other_weights = matrix[other_idx].toarray().ravel()
    shared = np.minimum(input_weights, other_weights)
    best = shared.argsort()[::-1][:top_terms]
    return ", ".join(feature_names[i] for i in best if shared[i] > 0)

def recommend_similar(title, similarity_matrix, matrix, vectorizer, n=5):
    idx = movies.index[movies["title"].eq(title)][0]
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    rows = []
    for other_idx, score in scores[1:n+1]:
        rows.append({
            "input_movie": title,
            "recommended_movie": movies.loc[other_idx, "title"],
            "similarity_score": round(float(score), 3),
            "shared_terms_features": shared_terms(idx, other_idx, matrix, vectorizer),
        })
    return pd.DataFrame(rows)

recommend_similar("Interstellar", tfidf_similarity, tfidf_matrix, tfidf_vectorizer)


,input_movie,recommended_movie,similarity_score,shared_terms_features
0,Interstellar,Gravity,0.201,"astronauts, survival, space, sci, fi, drama"
1,Interstellar,Toy Story,0.168,"new, family, adventure, space"
2,Interstellar,Paddington,0.154,"family, home, adventure"
3,Interstellar,The Martian,0.125,"survival, space, adventure, fi, sci"
4,Interstellar,Inception,0.119,"nolan, christopher, fi, sci"


Compare Bag-of-Words and TF-IDF. The rankings may be similar, but TF-IDF usually gives cleaner emphasis to distinctive content.


In [7]:
bow_results = recommend_similar("Interstellar", bow_similarity, bow_matrix, count_vectorizer, n=5)
tfidf_results = recommend_similar("Interstellar", tfidf_similarity, tfidf_matrix, tfidf_vectorizer, n=5)

comparison = bow_results[["recommended_movie", "similarity_score"]].rename(columns={"similarity_score": "bow_score"})
comparison["tfidf_movie"] = tfidf_results["recommended_movie"]
comparison["tfidf_score"] = tfidf_results["similarity_score"]
comparison


,recommended_movie,bow_score,tfidf_movie,tfidf_score
0,Gravity,0.316,Gravity,0.201
1,Toy Story,0.258,Toy Story,0.168
2,Paddington,0.230,Paddington,0.154
3,The Martian,0.219,The Martian,0.125
4,Inception,0.191,Inception,0.119


## What did we learn?

- Bag-of-Words creates count vectors from text.
- TF-IDF keeps the vector idea but gives more weight to distinctive terms.
- Cosine similarity turns text vectors into a ranked recommendation list.

Exercises:
1. Try the recommender with `Toy Story` or `Titanic`.
2. Add a new keyword to one movie and check whether the ranking changes.
